# 04 · Validate — occlusion + specificity vs human metalloenzymes + BindCraft-vs-RFdiffusion

**Standard slot:** *validate (in silico).* **For Project 10 this is the mechanism core:** model
**substrate occlusion** (does the binder block the carbapenem-access channel?), counter-test
**specificity vs human metalloenzymes** (safety), run the **head-to-head** between the two paradigms,
and the **rim-vs-distal epitope ablation** (a distal patch should bind but **not** occlude), with
publication-style figures (D3 part 2).

> **Binding ≠ inhibition.** Occlusion and specificity are in-silico *enrichment*. Only the
> nitrocefin/carbapenem kinetics assay (notebook 05) measures inhibition (IC50). No IC50 is produced
> here — that would be fabricated.

Needs `results/bindcraft_designs.csv` + `results/rfdiffusion_designs.csv` + `results/all_ranked.csv`
(from notebooks 02–03).

## Setup paths

In [ ]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## 1 · Head-to-head hit rate + interface energy

Compare the two paradigms on (a) all-layers **hit rate** and (b) the **interface-energy** (`rosetta_dG`)
distribution of survivors. A fair comparison filters both identically (notebook 03) and reports the
*distribution*, not the single best. Mock numbers are SYNTHETIC.

In [ ]:
import pandas as pd, numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

ranked = pd.read_csv("results/all_ranked.csv")
print("paradigms:", ranked["paradigm"].value_counts().to_dict())

summary = []
for p, g in ranked.groupby("paradigm"):
    n = len(g); passed = int((g["layers_passed"] >= 3).sum())
    summary.append(dict(paradigm=p, n=n, all_layers_survivors=passed,
                        hit_rate_pct=round(100*passed/max(n,1), 1),
                        median_pae=round(float(g["pae_interaction"].median()), 2),
                        median_dG=round(float(g["rosetta_dG"].median()), 2)))
summary = pd.DataFrame(summary)
print("\nhead-to-head summary (SYNTHETIC if mock):")
print(summary.to_string(index=False))

In [ ]:
# Interface-energy distribution per paradigm (survivors).
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
for p, g in ranked.groupby("paradigm"):
    surv = g[g["layers_passed"] >= 3]
    ax[0].hist(g["pae_interaction"].dropna(), bins=15, alpha=0.5, label=p)
    ax[1].hist(surv["rosetta_dG"].dropna(), bins=15, alpha=0.5, label=p)
ax[0].set_xlabel("pae_interaction (Å, lower better)"); ax[0].set_ylabel("designs"); ax[0].set_title("AF2-Multimer pae_interaction"); ax[0].legend()
ax[1].set_xlabel("rosetta_dG (REU, more negative better)"); ax[1].set_title("Interface energy (survivors)"); ax[1].legend()
fig.suptitle("BindCraft vs RFdiffusion vs NDM-1 (EXAMPLE_DATA if mock)")
plt.tight_layout(); plt.savefig("results/p10_headtohead.png", dpi=150); plt.show()
print("saved results/p10_headtohead.png")

## 2 · Substrate-occlusion modeling (mechanism #1) `[core]`

A binder only **inhibits** if it occludes the substrate-access channel over the di-zinc site. We
re-score every survivor with `occlusion_score()` (the mock proxy combines rim coverage with a
pocket-fit term; on Colab, replace with a pocket-volume / SASA with-vs-without-binder measurement and
optional carbapenem docking — see `binder_tools.occlusion_score(..., tool="pocket")`). Compare the
occlusion distributions across paradigms; an interface that does **not** occlude is a sticker, not an
inhibitor.

In [ ]:
import binder_tools as bt

bc = pd.read_csv("results/bindcraft_designs.csv")
rf = pd.read_csv("results/rfdiffusion_designs.csv")
pools = pd.concat([bc, rf], ignore_index=True)

# Join precomputed occlusion (from nb 02) onto the ranked survivors; recompute if absent.
if "occlusion" in pools.columns:
    occ_map = pools.set_index("design_id")["occlusion"]
    ranked["occlusion"] = ranked["design_id"].map(occ_map)
else:
    HOTSPOTS = bt.parse_hotspots("A120,A220,A228")
    ranked["occlusion"] = ranked.apply(
        lambda r: bt.occlusion_score(r.get("contact_residues", ""), HOTSPOTS, tool="mock")["occlusion"], axis=1)

OCCLUSION_MIN = 0.5      # project threshold: occludes >= half the substrate-access channel
surv = ranked[ranked["layers_passed"] >= 3].copy()
print("occlusion of all-layers survivors (SYNTHETIC if mock):")
for p, g in surv.groupby("paradigm"):
    occluders = int((g["occlusion"] >= OCCLUSION_MIN).sum())
    print(f"  {p:12s}: median occlusion = {g['occlusion'].median():.2f}  | occluders(>= {OCCLUSION_MIN}) = {occluders}/{len(g)}")

fig, ax = plt.subplots(figsize=(5.2, 3.4))
for p, g in surv.groupby("paradigm"):
    ax.hist(g["occlusion"].dropna(), bins=12, alpha=0.5, label=p)
ax.axvline(OCCLUSION_MIN, color="k", ls="--", lw=1, label=f"threshold {OCCLUSION_MIN}")
ax.set_xlabel("occlusion (0–1, higher = more channel blocked)"); ax.set_ylabel("survivors")
ax.set_title("Substrate occlusion (EXAMPLE_DATA if mock)"); ax.legend()
plt.tight_layout(); plt.savefig("results/p10_occlusion.png", dpi=150); plt.show()
print("saved results/p10_occlusion.png  —  occlusion != inhibition (the IC50 assay tests that, nb 05)")

## 3 · Specificity vs human metalloenzymes (mechanism #2) `[core]`

A di-zinc-site binder that also hits **human** Zn/metalloenzymes is a safety liability. Counter-test
each survivor against a small panel with `offtarget_specificity()` (on Colab: AF2-Multimer of the
binder vs each human enzyme; here a deterministic SYNTHETIC proxy). Higher specificity = more
NDM-1-selective. This is the in-silico mirror of the off-target-metalloenzyme **control** in the
assay (notebook 05).

In [ ]:
HUMAN_PANEL = ["CA2", "MMP9", "GLO2"]   # carbonic anhydrase II, MMP-9, glyoxalase II (human metalloenzymes)
SPECIFICITY_MIN = 0.5

# Build a tiny BinderDesign-like shim from each survivor row so offtarget_specificity() can read .sequence.
spec_rows = []
for _, r in surv.iterrows():
    seq = str(r.get("sequence", "")) or "A"
    # Worst-case (minimum) specificity across the panel = most conservative safety read.
    svals = []
    for enz in HUMAN_PANEL:
        # pass a lightweight object exposing sequence + design_id via a namespace
        shim = type("D", (), {"sequence": seq, "design_id": str(r["design_id"]), "pae_interaction": r.get("pae_interaction")})()
        svals.append(bt.offtarget_specificity(shim, enz, tool="mock")["specificity"])
    spec_rows.append(dict(design_id=str(r["design_id"]), paradigm=r["paradigm"],
                          min_specificity=round(min(svals), 3)))
spec_df = pd.DataFrame(spec_rows)
surv = surv.merge(spec_df[["design_id", "min_specificity"]], on="design_id", how="left")

print(f"specificity (min across {HUMAN_PANEL}) of survivors (SYNTHETIC if mock):")
for p, g in surv.groupby("paradigm"):
    selective = int((g["min_specificity"] >= SPECIFICITY_MIN).sum())
    print(f"  {p:12s}: median min-specificity = {g['min_specificity'].median():.2f} | selective(>= {SPECIFICITY_MIN}) = {selective}/{len(g)}")
print("\nHigh specificity = LOW predicted off-target binding to human metalloenzymes (safer).")

## 4 · Rim-vs-distal epitope ablation `[extension]`

The catalog ablation: **active-site rim vs a distal patch.** A binder steered to a **distal** surface
patch should still bind (decent `pae_interaction`) but **not** occlude the substrate channel (low
`occlusion`) — the cleanest demonstration that *epitope choice drives inhibition*. Here we scaffold it
by generating a distal-hotspot mock pool and comparing occlusion; on Colab, re-run the campaign with a
distal hotspot set and compare.

In [ ]:
# Scaffold: a distal-patch pool (different, non-active-site hotspots) for the ablation.
RIM_HOTSPOTS    = bt.parse_hotspots("A120,A220,A228")    # active-site rim (the inhibitory epitope)
DISTAL_HOTSPOTS = bt.parse_hotspots("A40,A55,A70")        # EXAMPLE distal patch — replace with a real distal surface

rim_pool    = bt.generate_binders_bindcraft("NDM1", RIM_HOTSPOTS, n=40, tool="mock")
distal_pool = bt.generate_binders_bindcraft("NDM1", DISTAL_HOTSPOTS, n=40, tool="mock")
bt.score_designs(rim_pool, tool="mock"); bt.score_designs(distal_pool, tool="mock")

def median_occ(designs, active_site):
    vals = [bt.occlusion_score(d, active_site, tool="mock")["occlusion"] for d in designs]
    return float(np.median(vals))

# Both are scored for occlusion AGAINST THE ACTIVE-SITE RIM (the thing that must be blocked to inhibit).
print("rim-vs-distal ablation (occlusion scored vs the active-site rim; SYNTHETIC):")
print(f"  rim-targeted binders   : median occlusion = {median_occ(rim_pool, RIM_HOTSPOTS):.2f}  (expected HIGHER)")
print(f"  distal-targeted binders: median occlusion = {median_occ(distal_pool, RIM_HOTSPOTS):.2f}  (expected LOWER)")
print("\nInterpretation: distal binders may stick but should NOT occlude the substrate channel ->")
print("epitope choice (active-site rim) is what makes a binder an inhibitor candidate, not just a sticker.")

## 5 · Select the top 10–20 per paradigm (occluding + selective)

The D★ deliverable wants the **top 10–20 each**. Rank survivors by the composite score and, as
mechanism tie-breakers, prefer **higher occlusion** then **higher specificity** — an inhibitor
candidate must both block the channel and spare human metalloenzymes. Save the shortlist for the
inhibition-assay plan (notebook 05).

In [ ]:
top_per = []
for p, g in surv.groupby("paradigm"):
    g2 = g.sort_values(["score", "occlusion", "min_specificity"], ascending=False).head(20)
    top_per.append(g2)
top = pd.concat(top_per, ignore_index=True)
top.to_csv("results/top_candidates.csv", index=False)
print("wrote results/top_candidates.csv:", top.shape, "(top<=20 per paradigm, occluding + selective)")
print(top.groupby("paradigm").size().to_dict())
cols = [c for c in ["design_id","paradigm","score","pae_interaction","rosetta_dG","occlusion","min_specificity"] if c in top.columns]
top.head(8)[cols]

## D3 (part 2) checklist
- [ ] Head-to-head: hit rate + interface-energy distribution per paradigm (figure `results/p10_headtohead.png`).
- [ ] **Occlusion** modeling of survivors (figure `results/p10_occlusion.png`); occluder count per paradigm.
- [ ] **Specificity vs human metalloenzymes** counter-test (min across the panel); selective count per paradigm.
- [ ] **Rim-vs-distal** ablation: distal binders bind but don't occlude — epitope choice drives inhibition.
- [ ] `results/top_candidates.csv`: top 10–20 each (occluding + selective), ready for the assay plan.
- [ ] Honest discussion: binding ≠ inhibition; the two paradigms' different failure modes (not just a winner).

**Next:** `05_validation_plan.ipynb` — the nitrocefin/carbapenem inhibition (IC50) plan.